# 04 - Evaluate Trained MLP on S3DIS Area 5

This notebook runs quantitative evaluation for the trained translation head on the pre-extracted S3DIS Area 5 features.

It expects feature `.npz` files and trained `.pth` checkpoints to already exist on the shared Google Drive.

### 1. Setup Repo & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
DRIVE_FEATURES_DIR = f'{DRIVE_ROOT}/features/s3dis_area5'
LOCAL_FEATURES_ROOT = '/content/local_features'
LOCAL_FEATURES_DIR = f'{LOCAL_FEATURES_ROOT}/s3dis_area5'
BRANCH = 'dev/eval-demo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


In [ ]:
%cd {REPO_DIR}
!git restore configs/train_mlp_s3dis.yaml notebooks/pyproject.toml
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull --no-edit origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

### 2. Wire Drive Paths and Copy Features Locally

We keep `data/` and `checkpoints/` on Drive, but copy `features/s3dis_area5` to local runtime storage for faster evaluation.

In [ ]:
%cd {REPO_DIR}
!rm -f data checkpoints features pretrained
!ln -sf {DRIVE_ROOT}/data ./data
!ln -sf {DRIVE_ROOT}/checkpoints ./checkpoints
!mkdir -p {LOCAL_FEATURES_ROOT}
!rm -rf {LOCAL_FEATURES_DIR}
!cp -r {DRIVE_FEATURES_DIR} {LOCAL_FEATURES_ROOT}/
!ln -sf {LOCAL_FEATURES_ROOT} ./features
!readlink -f ./features/s3dis_area5
!du -sh {LOCAL_FEATURES_DIR}
!ls -lh ./checkpoints/mlp_s3dis | tail

### 3. Setup Environment & Hugging Face Token

In [ ]:
%cd /content/Deep_learning_project/notebooks

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f'HF_TOKEN={HF_TOKEN}\n')

!uv python install 3.10.12
!uv sync --python 3.10.12

### 4. Choose a Checkpoint and Run Evaluation

Set `CHECKPOINT_NAME` to the checkpoint you want to evaluate, for example `epoch_040.pth` or `last_model.pth`.

In [ ]:
CONFIG_PATH = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
EVAL_BATCH_SIZE = 16384
CHECKPOINT_NAME = 'last_model.pth'
CHECKPOINT_PATH = f'/content/Deep_learning_project/checkpoints/mlp_s3dis/{CHECKPOINT_NAME}'
print('Config:', CONFIG_PATH)
print('Eval batch size:', EVAL_BATCH_SIZE)
print('Checkpoint:', CHECKPOINT_PATH)

!PYTHONPATH=/content/Deep_learning_project uv run python /content/Deep_learning_project/src/evaluate.py --config {CONFIG_PATH} --features_dir /content/Deep_learning_project/features/s3dis_area5 --checkpoint {CHECKPOINT_PATH} --batch_size {EVAL_BATCH_SIZE}